# FCC Fabric v8 — Served / Unserved Status (Cable & Fiber)

Classifies every location in the FCC Broadband Serviceable Location Fabric (v8) as **Served** or **Unserved** using the matching BDC fixed availability data.

**Rule**
- **Served** — at least one cable (tech 40) or fiber (tech 50) offering at or above **25/3 Mbps**.
- **Unserved** — everything else: no cable/fiber at all, cable/fiber only below 25/3, or no availability record.

**Mass market** — Fabric locations with `bsl_flag = True` (toggle in the config).

All processing runs in DuckDB; the output is a CSV with one row per Fabric location.

## 1. Configuration

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- Inputs
FABRIC_VERSION = "v8"

# Fabric location file (CSV or Parquet)
FABRIC_PATH = r"C:/data/fabric/FCC_Active_BSL_v8.csv"

# BDC availability files, one input per technology
# Each accepts a single file, a glob pattern, or a list of paths
CABLE_PATH = r"C:/data/bdc/availability/bdc_06_Cable_fixed_broadband.csv"
FIBER_PATH = r"C:/data/bdc/availability/bdc_06_FibertothePremises_fixed_broadband.csv"

# ---------------------------------------------------------------- Output
OUTPUT_DIR = Path(r"C:/data/output")
OUTPUT_CSV = OUTPUT_DIR / f"FFA_{FABRIC_VERSION}_served_unserved.csv"

# ---------------------------------------------------------------- Mass-market filter
# True  -> only mass-market locations (Fabric bsl_flag = True)
# False -> all locations in the Fabric file
FILTER_MASS_MARKET = True

# ---------------------------------------------------------------- Served threshold
MIN_DOWNLOAD_MBPS = 25
MIN_UPLOAD_MBPS = 3

# BDC technology codes counted toward "served"
TECH_CODES = {40: "Cable", 50: "Fiber"}

# ---------------------------------------------------------------- Column names
# Adjust if the source files use different headers
FAB_COLS = {
    "location_id": "location_id",
    "building_type": "building_type_code",   # used only in the summary table
    "bsl_flag": "bsl_flag",
}
AVAIL_COLS = {
    "location_id": "location_id",
    "technology": "technology",
    "download": "max_advertised_download_speed",
    "upload": "max_advertised_upload_speed",
    "provider": "provider_id",
}

# DuckDB resources (None = DuckDB defaults)
DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = None

# CSV parsing
# True  -> malformed rows are skipped and listed in a report after loading
# False -> any malformed row stops the run with an error
CSV_SKIP_BAD_ROWS = True

## 2. Setup

In [ ]:
import csv
import glob

import duckdb
import pandas as pd

con = duckdb.connect()
if DUCKDB_MEMORY_LIMIT:
    con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
if DUCKDB_THREADS:
    con.execute(f"SET threads = {DUCKDB_THREADS}")
con.execute("SET preserve_insertion_order = false")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def sql_list(values):
    """Formats a Python list as a SQL IN-list of quoted strings."""
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def reader(path):
    """Returns a DuckDB table function for a CSV/Parquet path, glob, or list of paths."""
    patterns = path if isinstance(path, (list, tuple)) else [path]
    files = []
    for pat in patterns:
        matches = sorted(glob.glob(str(pat)))
        if not matches:
            raise FileNotFoundError(f"No file matches: {pat}")
        files.extend(m.replace("\\", "/") for m in matches)
    src = "[" + ", ".join(f"'{f}'" for f in files) + "]"

    if all(f.lower().endswith(".parquet") for f in files):
        return f"read_parquet({src}, union_by_name = true)"

    # Reads the header row directly so DuckDB skips format detection,
    # which fails when a malformed row falls inside its sample
    headers = []
    for f in files:
        with open(f, newline="", encoding="utf-8-sig") as fh:
            headers.append(next(csv.reader(fh)))
    if any(h != headers[0] for h in headers[1:]):
        raise ValueError(f"Column headers differ between files: {files}")
    columns = "{" + ", ".join(f"'{c.strip()}': 'VARCHAR'" for c in headers[0]) + "}"

    opts = (f"header = true, auto_detect = false, columns = {columns}, "
            "delim = ',', quote = '\"', escape = '\"', null_padding = true")
    if CSV_SKIP_BAD_ROWS:
        opts += ", store_rejects = true"
    return f"read_csv({src}, {opts})"


def report_rejects(label):
    """Prints rows skipped by the CSV reader since the last call, then clears the log."""
    if not CSV_SKIP_BAD_ROWS:
        return
    try:
        rej = con.execute("""
            SELECT s.file_path, e.line, e.error_type, e.csv_line
            FROM reject_errors e JOIN reject_scans s USING (scan_id, file_id)
            ORDER BY s.file_path, e.line
        """).df()
    except duckdb.CatalogException:
        return
    rej = rej.drop_duplicates(subset=["file_path", "line"])
    if len(rej):
        print(f"WARNING [{label}]: {len(rej):,} malformed CSV row(s) skipped")
        with pd.option_context("display.max_colwidth", 120):
            print(rej.head(20).to_string(index=False))
    con.execute("DROP TABLE IF EXISTS reject_errors")
    con.execute("DROP TABLE IF EXISTS reject_scans")


# Fails early if any input path matches no file or headers differ
for label, p in [("Fabric", FABRIC_PATH), ("Cable", CABLE_PATH), ("Fiber", FIBER_PATH)]:
    reader(p)
    pats = p if isinstance(p, (list, tuple)) else [p]
    n = sum(len(glob.glob(str(x))) for x in pats)
    print(f"{label:<7} {n} file(s)")

## 3. Load Fabric locations
Applies the mass-market filter (bsl_flag = True) when enabled in the config.

In [ ]:
fab_where = ["1 = 1"]
if FILTER_MASS_MARKET:
    fab_where.append(f"lower(CAST({FAB_COLS['bsl_flag']} AS VARCHAR)) IN ('true', 't', '1', 'yes')")

con.execute(f"""
    CREATE OR REPLACE TABLE fabric AS
    SELECT
        TRY_CAST({FAB_COLS['location_id']} AS BIGINT) AS loc_id,
        *
    FROM {reader(FABRIC_PATH)}
    WHERE {' AND '.join(fab_where)}
""")

total_fabric = con.execute(f"SELECT count(*) FROM {reader(FABRIC_PATH)}").fetchone()[0]
kept_fabric = con.execute("SELECT count(*) FROM fabric").fetchone()[0]
report_rejects("Fabric")
bad_ids = con.execute("SELECT count(*) FROM fabric WHERE loc_id IS NULL").fetchone()[0]

print(f"Fabric rows in file:      {total_fabric:,}")
print(f"Fabric rows after filter: {kept_fabric:,}")
if bad_ids:
    print(f"WARNING: {bad_ids:,} rows have a non-numeric location_id")

if FAB_COLS["building_type"] in con.execute("SELECT * FROM fabric LIMIT 0").df().columns:
    con.sql(f"""
        SELECT {FAB_COLS['building_type']} AS building_type_code, count(*) AS locations
        FROM fabric GROUP BY 1 ORDER BY 2 DESC
    """).show()

## 4. Load availability (cable & fiber)
Reads the cable and fiber files separately and stacks them. Rows with a technology code outside `TECH_CODES` are dropped as a safeguard against a mislabeled file. All offerings count, residential and business.

In [ ]:
tech_in = ", ".join(str(t) for t in TECH_CODES)


def avail_select(path, source):
    """Returns a SELECT over one availability input, tagged with its source file type."""
    return f"""
        SELECT
            TRY_CAST({AVAIL_COLS['location_id']} AS BIGINT)   AS loc_id,
            TRY_CAST({AVAIL_COLS['technology']}  AS INTEGER)  AS technology,
            TRY_CAST({AVAIL_COLS['download']}    AS DOUBLE)   AS down_mbps,
            TRY_CAST({AVAIL_COLS['upload']}      AS DOUBLE)   AS up_mbps,
            {AVAIL_COLS['provider']}                          AS provider_id,
            '{source}'                                        AS source_file
        FROM {reader(path)}
    """


con.execute(f"""
    CREATE OR REPLACE TABLE avail_raw AS
    {avail_select(CABLE_PATH, 'cable')}
    UNION ALL
    {avail_select(FIBER_PATH, 'fiber')}
""")

report_rejects("Availability")

# Flags rows whose technology code does not match the file they came from
mismatch = con.execute("""
    SELECT source_file, technology, count(*) AS records
    FROM avail_raw
    WHERE (source_file = 'cable' AND technology IS DISTINCT FROM 40)
       OR (source_file = 'fiber' AND technology IS DISTINCT FROM 50)
    GROUP BY ALL ORDER BY ALL
""").df()
if len(mismatch):
    print("WARNING: technology codes not matching their input file:")
    print(mismatch.to_string(index=False))

con.execute(f"""
    CREATE OR REPLACE TABLE avail AS
    SELECT * EXCLUDE (source_file)
    FROM avail_raw
    WHERE technology IN ({tech_in})
""")

con.sql("""
    SELECT technology, count(*) AS records, count(DISTINCT loc_id) AS locations
    FROM avail GROUP BY 1 ORDER BY 1
""").show()

## 5. Summarize availability per location
A location qualifies as served when a **single** cable or fiber record meets both the download and upload threshold.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE loc_avail AS
    SELECT
        loc_id,
        bool_or(technology = 40)                                   AS has_cable,
        bool_or(technology = 50)                                   AS has_fiber,
        max(down_mbps)                                             AS max_cf_down_mbps,
        max(up_mbps)                                               AS max_cf_up_mbps,
        count(DISTINCT provider_id)
            FILTER (WHERE down_mbps >= {MIN_DOWNLOAD_MBPS}
                      AND up_mbps   >= {MIN_UPLOAD_MBPS})          AS qualifying_providers,
        bool_or(down_mbps >= {MIN_DOWNLOAD_MBPS}
            AND up_mbps   >= {MIN_UPLOAD_MBPS})                    AS meets_threshold
    FROM avail
    GROUP BY loc_id
""")
print(f"Locations with any cable/fiber record: {con.execute('SELECT count(*) FROM loc_avail').fetchone()[0]:,}")

## 6. Assign served / unserved status

In [ ]:
threshold = f"{MIN_DOWNLOAD_MBPS}/{MIN_UPLOAD_MBPS}"

con.execute(f"""
    CREATE OR REPLACE TABLE fabric_status AS
    SELECT
        f.* EXCLUDE (loc_id),
        coalesce(a.has_cable, false)               AS has_cable,
        coalesce(a.has_fiber, false)               AS has_fiber,
        a.max_cf_down_mbps,
        a.max_cf_up_mbps,
        coalesce(a.qualifying_providers, 0)        AS qualifying_providers,
        CASE WHEN coalesce(a.meets_threshold, false)
             THEN 'Served' ELSE 'Unserved' END     AS status,
        CASE
            WHEN coalesce(a.meets_threshold, false) THEN 'Cable/fiber at or above {threshold}'
            WHEN a.loc_id IS NOT NULL               THEN 'Cable/fiber below {threshold}'
            ELSE 'No cable/fiber'
        END                                        AS status_reason,
        '{FABRIC_VERSION}'                         AS fabric_version
    FROM fabric f
    LEFT JOIN loc_avail a USING (loc_id)
""")

con.sql("""
    SELECT status, status_reason, count(*) AS locations,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM fabric_status
    GROUP BY ALL
    ORDER BY status, locations DESC
""").show()

## 7. QA checks

In [ ]:
# Every filtered Fabric location appears exactly once in the output
n_out = con.execute("SELECT count(*) FROM fabric_status").fetchone()[0]
assert n_out == kept_fabric, f"Row count changed: {kept_fabric:,} -> {n_out:,}"

# Availability records whose location_id is missing from the filtered Fabric
orphans = con.execute("""
    SELECT count(DISTINCT loc_id) FROM avail
    WHERE loc_id NOT IN (SELECT loc_id FROM fabric WHERE loc_id IS NOT NULL)
""").fetchone()[0]
print(f"Output rows: {n_out:,} (matches filtered Fabric)")
print(f"Cable/fiber location_ids not in the filtered Fabric: {orphans:,}")
if orphans:
    print("  -> Expected if the Fabric filter removes locations, or if availability "
          "comes from a different Fabric vintage.")

# Status by county, when a county column exists
county_col = next((c for c in ["county_geoid", "county_fips", "county"]
                   if c in con.execute("SELECT * FROM fabric_status LIMIT 0").df().columns), None)
if county_col:
    con.sql(f"""
        SELECT {county_col},
               count(*) FILTER (WHERE status = 'Served')   AS served,
               count(*) FILTER (WHERE status = 'Unserved') AS unserved,
               round(100.0 * count(*) FILTER (WHERE status = 'Unserved') / count(*), 1) AS pct_unserved
        FROM fabric_status GROUP BY 1 ORDER BY unserved DESC LIMIT 15
    """).show()

## 8. Export CSV

In [ ]:
out = str(OUTPUT_CSV).replace("\\", "/")
con.execute(f"COPY fabric_status TO '{out}' (HEADER, DELIMITER ',')")
print(f"Wrote {n_out:,} rows -> {OUTPUT_CSV}")